# Competitive Insights

This notebook is the first analytical layer on top of the DuckDB warehouse.

It now starts with two visual explanations:

- how the repo moves from raw scraping output to analysis-ready data
- how the warehouse tables relate to each other

After the diagrams, the notebook keeps the analytical cuts by `platform + brand`,
`zone + platform`, and `product term + platform`.


## How The Diagrams Render In Notebook

There are a few valid ways to show Mermaid-like diagrams in Jupyter environments:

- native Mermaid markdown rendering in JupyterLab-compatible frontends
- Python packages that render Mermaid diagrams through external services
- inline HTML/Javascript rendering inside notebook outputs

This notebook uses the third option as the active renderer so the diagrams can appear
inline in notebook frontends that support HTML output. The diagram definitions themselves
still use Mermaid syntax.


In [1]:

from html import escape
from pathlib import Path
from uuid import uuid4

import duckdb
import pandas as pd

try:
    from IPython.display import HTML, display
    IPYTHON_DISPLAY = True
except ImportError:
    IPYTHON_DISPLAY = False

    class HTML(str):
        pass

    def display(*args, **kwargs):
        return None

try:
    import plotly.express as px
except ImportError:
    class _DummyFigure:
        def update_layout(self, *args, **kwargs):
            return self

        def show(self, *args, **kwargs):
            return None

    class _DummyPX:
        def imshow(self, *args, **kwargs):
            return _DummyFigure()

        def bar(self, *args, **kwargs):
            return _DummyFigure()

    px = _DummyPX()

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 200)


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "warehouse" / "intel_rappi.duckdb").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root with data/warehouse/intel_rappi.duckdb")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
DB_PATH = REPO_ROOT / "data" / "warehouse" / "intel_rappi.duckdb"
con = duckdb.connect(str(DB_PATH), read_only=True)


def q(sql: str) -> pd.DataFrame:
    return con.execute(sql).df()


def render_mermaid(graph_definition: str, *, height: int = 600) -> None:
    """Render Mermaid syntax inline in notebook frontends that allow HTML output."""
    if not IPYTHON_DISPLAY:
        print(graph_definition)
        return

    element_id = f"mermaid-{uuid4().hex}"
    vp_id = f"{element_id}-viewport"
    inner_id = f"{element_id}-inner"
    html = f"""
    <div id="{element_id}" style="border:1px solid #e5e7eb; border-radius:12px; overflow:hidden; margin:8px 0; background:#fcfcfd; color:#111827;">
      <div style="display:flex;flex-wrap:wrap;align-items:center;gap:8px;padding:8px 12px;border-bottom:1px solid #e5e7eb;font-size:12px;color:#374151;background:#f8fafc;">
        <span style="font-weight:600;margin-right:4px;">Diagram</span>
        <button type="button" id="{element_id}-zi" style="cursor:pointer;padding:2px 10px;border:1px solid #cbd5e1;border-radius:6px;background:#fff;">+</button>
        <button type="button" id="{element_id}-zo" style="cursor:pointer;padding:2px 10px;border:1px solid #cbd5e1;border-radius:6px;background:#fff;">−</button>
        <button type="button" id="{element_id}-zr" style="cursor:pointer;padding:2px 10px;border:1px solid #cbd5e1;border-radius:6px;background:#fff;">Reset</button>
        <span style="opacity:0.9;">Ctrl/⌘ + scroll to zoom · drag to pan</span>
      </div>
      <div id="{vp_id}" style="height:{height}px;overflow:hidden;position:relative;cursor:grab;touch-action:none;background:#fcfcfd;user-select:none;">
        <div id="{inner_id}" style="width:100%;height:100%;box-sizing:border-box;">
          <pre class="mermaid">{escape(graph_definition)}</pre>
        </div>
      </div>
    </div>
    <script>
    (function() {{
      const root = document.getElementById("{element_id}");
      const viewport = document.getElementById("{vp_id}");
      const inner = document.getElementById("{inner_id}");
      if (!root || !viewport || !inner) return;

      let svg = null;
      let baseVB = null;
      let vb = null;
      const MIN_W_FRAC = 0.12;
      const MAX_W_FRAC = 6;

      function parseVB(s) {{
        const p = (s || "").trim().split(/[\\s,]+/).map(Number);
        if (p.length !== 4 || p.some(function(x) {{ return !isFinite(x); }})) return null;
        return {{ x: p[0], y: p[1], w: p[2], h: p[3] }};
      }}

      function readSvgViewBox(s) {{
        if (s.viewBox && s.viewBox.baseVal && s.viewBox.baseVal.width > 0) {{
          const b = s.viewBox.baseVal;
          return {{ x: b.x, y: b.y, w: b.width, h: b.height }};
        }}
        const attr = s.getAttribute("viewBox");
        const parsed = parseVB(attr);
        if (parsed) return parsed;
        try {{
          const bb = s.getBBox();
          if (bb.width > 0 && bb.height > 0) return {{ x: bb.x, y: bb.y, w: bb.width, h: bb.height }};
        }} catch (e) {{}}
        return null;
      }}

      function applyVB() {{
        if (!svg || !vb) return;
        svg.setAttribute("viewBox", vb.x + " " + vb.y + " " + vb.w + " " + vb.h);
      }}

      function screenToSvg(s, clientX, clientY) {{
        const pt = s.createSVGPoint();
        pt.x = clientX;
        pt.y = clientY;
        const ctm = s.getScreenCTM();
        if (!ctm) return null;
        return pt.matrixTransform(ctm.inverse());
      }}

      function zoomAtPoint(zoomFactor, clientX, clientY) {{
        if (!svg || !vb || !baseVB) return;
        const pt = screenToSvg(svg, clientX, clientY);
        if (!pt) return;
        const minW = baseVB.w * MIN_W_FRAC;
        const maxW = baseVB.w * MAX_W_FRAC;
        let nw = vb.w / zoomFactor;
        nw = Math.min(maxW, Math.max(minW, nw));
        const nh = nw * (vb.h / vb.w);
        const sx = nw / vb.w;
        const sy = nh / vb.h;
        vb.x = pt.x - (pt.x - vb.x) * sx;
        vb.y = pt.y - (pt.y - vb.y) * sy;
        vb.w = nw;
        vb.h = nh;
        applyVB();
      }}

      function setupSvgPanzoom() {{
        svg = inner.querySelector("svg");
        if (!svg) return false;
        svg.style.width = "100%";
        svg.style.height = "100%";
        svg.style.display = "block";
        svg.setAttribute("preserveAspectRatio", "xMidYMid meet");

        const initial = readSvgViewBox(svg);
        if (!initial) return false;
        if (!svg.getAttribute("viewBox")) {{
          svg.setAttribute("viewBox", initial.x + " " + initial.y + " " + initial.w + " " + initial.h);
        }}
        baseVB = {{ x: initial.x, y: initial.y, w: initial.w, h: initial.h }};
        vb = {{ x: baseVB.x, y: baseVB.y, w: baseVB.w, h: baseVB.h }};
        applyVB();

        viewport.addEventListener("wheel", function(e) {{
          if (!(e.ctrlKey || e.metaKey)) return;
          e.preventDefault();
          const zf = e.deltaY < 0 ? 1.08 : 1 / 1.08;
          zoomAtPoint(zf, e.clientX, e.clientY);
        }}, {{ passive: false }});

        let dragging = false;
        let sx = 0, sy = 0, vbx0 = 0, vby0 = 0;
        viewport.addEventListener("pointerdown", function(e) {{
          if (e.button !== 0) return;
          dragging = true;
          sx = e.clientX;
          sy = e.clientY;
          vbx0 = vb.x;
          vby0 = vb.y;
          viewport.setPointerCapture(e.pointerId);
          viewport.style.cursor = "grabbing";
        }});
        viewport.addEventListener("pointermove", function(e) {{
          if (!dragging || !vb) return;
          const dx = e.clientX - sx;
          const dy = e.clientY - sy;
          const rw = viewport.clientWidth || 1;
          const rh = viewport.clientHeight || 1;
          vb.x = vbx0 - dx * (vb.w / rw);
          vb.y = vby0 - dy * (vb.h / rh);
          applyVB();
        }});
        function endDrag(e) {{
          if (!dragging) return;
          dragging = false;
          viewport.style.cursor = "grab";
          try {{ viewport.releasePointerCapture(e.pointerId); }} catch (err) {{}}
        }}
        viewport.addEventListener("pointerup", endDrag);
        viewport.addEventListener("pointercancel", endDrag);

        const zi = document.getElementById("{element_id}-zi");
        const zo = document.getElementById("{element_id}-zo");
        const zr = document.getElementById("{element_id}-zr");
        if (zi) zi.addEventListener("click", function() {{
          const r = viewport.getBoundingClientRect();
          zoomAtPoint(1.15, r.left + r.width / 2, r.top + r.height / 2);
        }});
        if (zo) zo.addEventListener("click", function() {{
          const r = viewport.getBoundingClientRect();
          zoomAtPoint(1 / 1.15, r.left + r.width / 2, r.top + r.height / 2);
        }});
        if (zr) zr.addEventListener("click", function() {{
          if (baseVB) vb = {{ x: baseVB.x, y: baseVB.y, w: baseVB.w, h: baseVB.h }};
          applyVB();
        }});
        return true;
      }}

      function boot() {{
        if (!window.mermaid) return;
        window.mermaid.initialize({{
          startOnLoad: false,
          theme: "neutral",
          securityLevel: "loose"
        }});
        const nodes = inner.querySelectorAll(".mermaid");
        if (nodes.length > 0) {{
          Promise.resolve(window.mermaid.init(undefined, nodes)).then(function() {{
            requestAnimationFrame(function() {{
              if (!setupSvgPanzoom()) {{
                console.warn("Mermaid pan-zoom: no SVG found");
              }}
            }});
          }}).catch(function() {{
            setupSvgPanzoom();
          }});
        }}
      }}

      if (!window.mermaid) {{
        const script = document.createElement("script");
        script.src = "https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.min.js";
        script.onload = boot;
        document.head.appendChild(script);
      }} else {{
        boot();
      }}
    }})();
    </script>
    """
    display(HTML(html))
print(f"Using warehouse: {DB_PATH}")


Using warehouse: /Users/jumafe/Desktop/personal_things/intel-rappi/data/warehouse/intel_rappi.duckdb


## 1. Workflow: From Scraping To Analysis-Ready Data

This diagram explains the end-to-end workflow used in the repo to transform raw platform
outputs into clean analytical tables.


In [2]:
workflow_mermaid = r"""
flowchart TB
    subgraph SCRAPE[1. Raw Collection]
        A[scripts/run_rappi_scraper.py] --> A1[rappi_<run_id>.json]
        B[scripts/run_uber_scraper.py] --> B1[uber_eats_<run_id>.json]
        C[scripts/run_didi_scraper.py] --> C1[didi_food_<run_id>.json]
    end

    subgraph RAW[2. Raw Landing Zone]
        D[data/raw/*.json]
    end

    subgraph BUILD[3. Warehouse Build]
        E[scripts/build_warehouse.py]
        E1[discover latest raw file per platform]
        E2[validate top-level structure]
        E3[validate snapshot-level fields]
        E4[normalize types and zone categories]
        E5[generate snapshot_id and derived flags]
        E6[explode products_matched into product_matches rows]
        E7[run quality checks]
    end

    subgraph WAREHOUSE[4. Analytical Storage]
        F[(intel_rappi.duckdb)]
        F1[runs]
        F2[snapshots]
        F3[product_matches]
        F4[zones_dim]
        F5[brands_dim]
        F6[data_quality_checks]
        F7[parquet exports]
    end

    subgraph ANALYSIS[5. Analysis Layer]
        G[notebooks/competitive_insights.ipynb]
    end

    A1 --> D
    B1 --> D
    C1 --> D
    D --> E
    E --> E1 --> E2 --> E3 --> E4 --> E5 --> E6 --> E7 --> F
    F --> F1
    F --> F2
    F --> F3
    F --> F4
    F --> F5
    F --> F6
    F --> F7
    F --> G

"""

render_mermaid(workflow_mermaid, height=500)


### Workflow Notes

- The 3 platform scripts produce comparable raw JSON files, but they are still operational outputs.
- `build_warehouse.py` is the cleaning and normalization step.
- The warehouse is where raw observations become structured facts, dimensions, and quality checks.
- The notebook should always read from DuckDB, not directly from the raw JSON files.


## 2. Warehouse Table Relationships

This diagram focuses on table relationships and the key analytical columns.
The full column dictionary appears immediately below.


In [3]:
schema_mermaid = r"""
erDiagram
    RUNS {
        string platform PK
        string run_id UK
        timestamp started_at
        timestamp completed_at
        bigint total_snapshots
        bigint successful
        bigint failed
        string raw_file
    }

    SNAPSHOTS {
        string snapshot_id PK
        string run_id FK
        string platform FK
        string brand_key FK
        string zone_name FK
        string restaurant_brand
        string store_id
        string store_name
        double delivery_fee
        double service_fee_pct
        double eta_minutes
        double rating
        bigint discount_count
        bigint products_total
        bigint matched_items_total
        boolean is_successful
        boolean has_product_price
        boolean has_delivery_fee
        boolean has_service_fee
        boolean has_eta
        boolean has_discount_signal
        boolean has_availability
        boolean has_final_total
    }

    PRODUCT_MATCHES {
        string snapshot_id FK
        string run_id FK
        string platform
        string brand_key FK
        string restaurant_brand
        string zone_name
        string search_term
        bigint match_rank
        string product_name
        string section
        double price
        string currency
    }

    ZONES_DIM {
        string zone_name PK
        string zone_category
        string zone_category_raw
        double zone_lat
        double zone_lng
        string zone_address
    }

    BRANDS_DIM {
        string brand_key PK
        string restaurant_brand
        string platform_id_rappi
        string platform_id_uber_eats
        string platform_id_didi_food
        bigint tracked_products_count
        bigint tracked_search_terms_count
    }

    DATA_QUALITY_CHECKS {
        string platform FK
        string run_id FK
        string check_name
        string severity
        string status
        double observed_value
        double expected_value
        string details
    }

    RUNS ||--o{ SNAPSHOTS : contains
    RUNS ||--o{ DATA_QUALITY_CHECKS : audits
    SNAPSHOTS ||--o{ PRODUCT_MATCHES : expands_to
    ZONES_DIM ||--o{ SNAPSHOTS : contextualizes
    BRANDS_DIM ||--o{ SNAPSHOTS : standardizes
    BRANDS_DIM ||--o{ PRODUCT_MATCHES : labels
"""

render_mermaid(schema_mermaid, height=1100)


## 3. Comparative Scope And Metric Coverage

This section defines what can actually be benchmarked with the current warehouse.

- `Rappi vs Uber Eats`: pricing, ETA, service-fee visibility, promo intensity, availability
- `Rappi vs DiDi Food`: pricing, promo intensity, availability
- `Rappi vs Uber Eats vs DiDi Food`: strongest on comparable product pricing and visible promotion signals

The goal is to keep the notebook decision-oriented: compare only where the current data supports a fair comparison, and call out the blind spots explicitly.


In [4]:
CANONICAL_TERM_SQL = """
case
    when lower(search_term) like 'cuarto%' then 'Cuarto de Libra'
    else search_term
end
"""

PLATFORM_ORDER = ["rappi", "uber_eats", "didi_food"]
PLATFORM_LABELS = {
    "rappi": "Rappi",
    "uber_eats": "Uber Eats",
    "didi_food": "DiDi Food",
}
PEER_LABELS = {
    "uber_eats": "Uber Eats",
    "didi_food": "DiDi Food",
}
COVERAGE_COLUMNS = [
    ("price_cov_pct", "Product Price"),
    ("delivery_cov_pct", "Delivery Fee"),
    ("service_cov_pct", "Service Fee"),
    ("eta_cov_pct", "ETA"),
    ("discount_cov_pct", "Discount Signal"),
    ("availability_cov_pct", "Availability"),
]

def pretty_platform(value: str) -> str:
    return PLATFORM_LABELS.get(value, value)

def pretty_peer(value: str) -> str:
    return PEER_LABELS.get(value, value)

def show_frame(frame: pd.DataFrame) -> None:
    if IPYTHON_DISPLAY:
        display(frame)
    else:
        print(frame.to_string(index=False))

def show_note(title: str, bullets: list[str]) -> None:
    if IPYTHON_DISPLAY:
        items = "".join(f"<li>{escape(item)}</li>" for item in bullets)
        # Explicit colors so readouts stay readable in dark-themed notebooks (light text on light bg otherwise).
        html = (
            "<div style=\"border:1px solid #e5e7eb; border-radius:12px; padding:14px; margin:12px 0; "
            "background:#fcfcfd; color:#111827; line-height:1.55; font-size:14px;\">"
            f"<strong style=\"color:#0f172a;\">{escape(title)}</strong>"
            f"<ul style=\"margin:8px 0 0 0; padding-left:1.25rem;\">{items}</ul></div>"
        )
        display(HTML(html))
    else:
        print(title)
        for item in bullets:
            print(f"- {item}")

def show_figure(fig) -> None:
    if IPYTHON_DISPLAY:
        fig.show()

def show_insight(title: str, finding: str, impact: str, recommendation: str, extra: list[str] | None = None) -> None:
    bullets = [
        f"Finding: {finding}",
        f"Impact: {impact}",
        f"Recommendation: {recommendation}",
    ]
    if extra:
        bullets.extend(extra)
    show_note(title, bullets)

def join_values(values: list[str]) -> str:
    cleaned = [str(value) for value in values if pd.notna(value) and str(value).strip()]
    cleaned = list(dict.fromkeys(cleaned))
    if not cleaned:
        return "none"
    if len(cleaned) == 1:
        return cleaned[0]
    return ", ".join(cleaned[:-1]) + f" and {cleaned[-1]}"

def classify_position(avg_pct_gap: float, threshold: float = 5.0) -> str:
    if pd.isna(avg_pct_gap):
        return "No data"
    if avg_pct_gap <= -threshold:
        return "Rappi cheaper"
    if avg_pct_gap >= threshold:
        return "Rappi pricier"
    return "Similar"

metric_coverage = q("""
select platform,
       count(*) as snapshots,
       round(avg(case when has_product_price then 1 else 0 end) * 100, 1) as price_cov_pct,
       round(avg(case when has_delivery_fee then 1 else 0 end) * 100, 1) as delivery_cov_pct,
       round(avg(case when has_service_fee then 1 else 0 end) * 100, 1) as service_cov_pct,
       round(avg(case when has_eta then 1 else 0 end) * 100, 1) as eta_cov_pct,
       round(avg(case when has_discount_signal then 1 else 0 end) * 100, 1) as discount_cov_pct,
       round(avg(case when has_availability then 1 else 0 end) * 100, 1) as availability_cov_pct
from snapshots
group by 1
order by 1
""")

metric_coverage["metrics_with_observed_coverage"] = metric_coverage[[name for name, _ in COVERAGE_COLUMNS]].gt(0).sum(axis=1)
coverage_display = metric_coverage.copy()
coverage_display["platform"] = coverage_display["platform"].map(pretty_platform)
show_frame(coverage_display)

coverage_heatmap = metric_coverage.set_index("platform")[[name for name, _ in COVERAGE_COLUMNS]]
coverage_heatmap.index = [pretty_platform(value) for value in coverage_heatmap.index]
coverage_heatmap.columns = [label for _, label in COVERAGE_COLUMNS]
coverage_fig = px.imshow(coverage_heatmap, aspect="auto", color_continuous_scale="Blues", labels={"color": "Coverage %"})
coverage_fig.update_layout(title="Observed Metric Coverage by Platform (%)", height=380)
show_figure(coverage_fig)

scope_bullets = []
for _, row in metric_coverage.iterrows():
    observed = [label for column, label in COVERAGE_COLUMNS if row[column] > 0]
    scope_bullets.append(
        f"{pretty_platform(row['platform'])} has observed coverage for {row['metrics_with_observed_coverage']}/6 core metrics: {join_values(observed)}."
    )
show_note("Scope from the current warehouse", scope_bullets)
show_insight(
    "Coverage as operational advantage",
    "Rappi is the only platform with observed coverage across the 6 core benchmarking metrics, while Uber Eats covers 5/6 and DiDi Food 3/6.",
    "That gives Rappi the broadest visibility to manage price, ETA, fees, promotions, and availability together. If the signal is operationalized well, Rappi can react faster than competitors by zone instead of making generic citywide decisions.",
    "Protect this data advantage as a product asset: keep extraction coverage stable, close the remaining availability gaps, and use the richer signal to prioritize interventions in dispatch, pricing, and promotions.",
)




,platform,snapshots,price_cov_pct,delivery_cov_pct,service_cov_pct,eta_cov_pct,discount_cov_pct,availability_cov_pct,metrics_with_observed_coverage
0,DiDi Food,44,97.7,0.0,0.0,0.0,100.0,100.0,3
1,Rappi,44,47.7,93.2,93.2,93.2,100.0,93.2,6
2,Uber Eats,40,62.5,0.0,100.0,95.0,100.0,100.0,5


## 4. Price Positioning

To avoid unfair comparisons, prices are benchmarked at the most comparable grain available in the warehouse: `brand + zone + normalized tracked product term`.

Because each scraper can return more than one matched item per term, the notebook uses the **lowest observed matched price per platform/zone/term** as the closest customer-facing comparable offer.


In [5]:
product_term = q(f"""
with canonical as (
    select platform, restaurant_brand, zone_name,
           {CANONICAL_TERM_SQL} as canonical_term,
           min(price) as snapshot_price
    from product_matches
    group by 1, 2, 3, 4
), median_prices as (
    select platform, restaurant_brand, canonical_term,
           count(*) as comparable_snapshots,
           round(avg(snapshot_price), 2) as avg_snapshot_price,
           round(median(snapshot_price), 2) as median_snapshot_price
    from canonical
    group by 1, 2, 3
)
select *
from median_prices
order by restaurant_brand, canonical_term, platform
""")

product_term["platform_label"] = product_term["platform"].map(pretty_platform)
price_matrix = product_term.pivot_table(
    index=["restaurant_brand", "canonical_term"],
    columns="platform_label",
    values="median_snapshot_price",
    aggfunc="first",
)
price_matrix = price_matrix.reindex(columns=["Rappi", "Uber Eats", "DiDi Food"])
price_matrix.index = [f"{brand} | {term}" for brand, term in price_matrix.index]
show_frame(price_matrix.reset_index().rename(columns={"index": "brand_term"}))

price_fig = px.imshow(price_matrix, aspect="auto", color_continuous_scale="Oranges", labels={"color": "Median comparable price (MXN)"})
price_fig.update_layout(title="Median Comparable Price by Brand / Product Term", height=520)
show_figure(price_fig)

platform_brand = q(f"""
with canonical as (
    select platform, restaurant_brand, zone_name,
           {CANONICAL_TERM_SQL} as canonical_term,
           min(price) as snapshot_price
    from product_matches
    group by 1, 2, 3, 4
), pairwise as (
    select 'uber_eats' as peer, r.restaurant_brand, r.zone_name, r.canonical_term,
           r.snapshot_price as rappi_price, u.snapshot_price as peer_price,
           r.snapshot_price - u.snapshot_price as gap_mxn,
           100.0 * (r.snapshot_price - u.snapshot_price) / nullif(u.snapshot_price, 0) as pct_gap
    from canonical r
    join canonical u using (restaurant_brand, zone_name, canonical_term)
    where r.platform = 'rappi' and u.platform = 'uber_eats'
    union all
    select 'didi_food' as peer, r.restaurant_brand, r.zone_name, r.canonical_term,
           r.snapshot_price as rappi_price, d.snapshot_price as peer_price,
           r.snapshot_price - d.snapshot_price as gap_mxn,
           100.0 * (r.snapshot_price - d.snapshot_price) / nullif(d.snapshot_price, 0) as pct_gap
    from canonical r
    join canonical d using (restaurant_brand, zone_name, canonical_term)
    where r.platform = 'rappi' and d.platform = 'didi_food'
)
select peer, restaurant_brand, canonical_term,
       count(*) as comparisons,
       round(avg(gap_mxn), 2) as avg_gap_mxn,
       round(median(gap_mxn), 2) as median_gap_mxn,
       round(avg(pct_gap), 2) as avg_pct_gap,
       round(median(pct_gap), 2) as median_pct_gap
from pairwise
group by 1, 2, 3
order by 1, 2, 3
""")

platform_brand["peer_label"] = platform_brand["peer"].map(pretty_peer)
platform_brand["positioning"] = platform_brand["avg_pct_gap"].apply(classify_position)
price_summary_display = platform_brand[[
    "peer_label", "restaurant_brand", "canonical_term", "comparisons",
    "avg_gap_mxn", "avg_pct_gap", "positioning"
]].rename(columns={"peer_label": "peer"})
show_frame(price_summary_display)

price_rank = product_term.copy()
price_rank["price_rank"] = price_rank.groupby(["restaurant_brand", "canonical_term"])["median_snapshot_price"].rank(method="dense")
cheapest_or_tied = price_rank[price_rank["price_rank"] == 1].groupby("platform_label").size().to_dict()

uber_terms = platform_brand[platform_brand["peer"] == "uber_eats"]
didi_terms = platform_brand[platform_brand["peer"] == "didi_food"]


price_examples = [
    ("Domino's Pizza | Combo", "Domino's Pizza Combo"),
    ("Domino's Pizza | Mediana", "Domino's Pizza Mediana"),
    ("McDonald's | Big Mac", "McDonald's Big Mac"),
]
price_example_lines = []
for key, label in price_examples:
    row = price_matrix.loc[key]
    price_example_lines.append(
        f"{label}: Rappi {row['Rappi']:.2f} MXN vs Uber Eats {row['Uber Eats']:.2f} MXN vs DiDi Food {row['DiDi Food']:.2f} MXN."
    )

show_insight(
    "Price positioning insight",
    "Rappi holds clearly competitive price points in Domino's Pizza Combo (159 MXN vs 159 Uber Eats vs 179 DiDi Food) and Domino's Pizza Mediana (169 MXN vs 169 Uber Eats vs 179 DiDi Food). For McDonald's Big Mac, Rappi still beats DiDi Food (113 MXN vs 125) but trails Uber Eats (99), so the competitive pricing playbook is working unevenly by item.",
    "When Rappi matches or beats peer prices on high-visibility anchor items, users have fewer reasons to switch platforms on price alone. That protects conversion and gives Rappi more room to compete on speed and promos instead of discounts everywhere.",
    "Keep the pricing mechanics that make Domino's Combo and Mediana competitive, document how those price points are being achieved, and replicate that logic on items where Rappi still trails Uber Eats, especially McDonald's anchor products.",
    extra=price_example_lines + ["Warehouse note: there is no directly comparable McDonald's Combo term in the tracked matched products, so the McDonald's benchmark uses the comparable anchor items Big Mac, Cuarto de Libra, and McNuggets instead."],
)


platform_label,brand_term,Rappi,Uber Eats,DiDi Food
0,Domino's Pizza | Combo,159.00,159.0,179.0
1,Domino's Pizza | Grande,209.05,199.0,209.0
2,Domino's Pizza | Mediana,169.00,169.0,179.0
3,McDonald's | Big Mac,113.00,99.0,125.0
4,McDonald's | Cuarto de Libra,135.00,135.0,135.0
5,McDonald's | McNuggets,99.00,59.0,59.0


,peer,restaurant_brand,canonical_term,comparisons,avg_gap_mxn,avg_pct_gap,positioning
0,DiDi Food,Domino's Pizza,Combo,17,-14.12,-7.89,Rappi cheaper
1,DiDi Food,Domino's Pizza,Grande,17,23.84,11.19,Rappi pricier
2,DiDi Food,Domino's Pizza,Mediana,17,-8.82,-4.67,Similar
3,DiDi Food,McDonald's,Big Mac,4,-12.00,-9.60,Rappi cheaper
4,DiDi Food,McDonald's,Cuarto de Libra,4,0.00,0.00,Similar
5,DiDi Food,McDonald's,McNuggets,4,40.00,67.80,Rappi pricier
6,Uber Eats,Domino's Pizza,Combo,15,4.00,2.52,Similar
7,Uber Eats,Domino's Pizza,Grande,15,26.82,13.74,Rappi pricier
8,Uber Eats,Domino's Pizza,Mediana,15,4.67,2.76,Similar
9,Uber Eats,McDonald's,Big Mac,2,14.00,14.14,Rappi pricier


## 5. Operational Advantage / Disadvantage

Operational benchmarking is currently strongest for `Rappi vs Uber Eats`, because DiDi Food does not expose ETA(Estimated Time of Arrival) in the warehouse snapshot set.


In [6]:
eta_summary = q("""
select platform,
       count(*) filter (where has_eta) as eta_snapshots,
       round(avg(eta_minutes), 2) as avg_eta_minutes,
       round(median(eta_minutes), 2) as median_eta_minutes
from snapshots
group by 1
order by 1
""")
eta_summary["platform_label"] = eta_summary["platform"].map(pretty_platform)
show_frame(eta_summary)

eta_fig = px.bar(
    eta_summary.dropna(subset=["median_eta_minutes"]),
    x="platform_label",
    y="median_eta_minutes",
    color="platform_label",
    labels={"platform_label": "Platform", "median_eta_minutes": "Median ETA (min)"},
)
eta_fig.update_layout(title="Median ETA by Platform", height=360, showlegend=False)
show_figure(eta_fig)

eta_zone_gap = q("""
with latest as (
    select *,
           row_number() over (partition by platform, zone_name, restaurant_brand order by scraped_at desc nulls last) as rn
    from snapshots
), filtered as (
    select *
    from latest
    where rn = 1
)
select zone_name,
       round(avg(case when platform = 'rappi' then eta_minutes end), 2) as rappi_eta,
       round(avg(case when platform = 'uber_eats' then eta_minutes end), 2) as uber_eta,
       round(avg(case when platform = 'rappi' then eta_minutes end) - avg(case when platform = 'uber_eats' then eta_minutes end), 2) as eta_gap_rappi_vs_uber
from filtered
group by 1
order by eta_gap_rappi_vs_uber desc nulls last, zone_name
""")
show_frame(eta_zone_gap.dropna(subset=["eta_gap_rappi_vs_uber"]).head(10))

top_eta_lag_zones = eta_zone_gap.dropna(subset=["eta_gap_rappi_vs_uber"]).head(3)["zone_name"].tolist()
avg_eta_gap = round(eta_zone_gap["eta_gap_rappi_vs_uber"].dropna().mean(), 2)
median_eta_gap = round(eta_zone_gap["eta_gap_rappi_vs_uber"].dropna().median(), 2)


show_insight(
    "Operational insight",
    f"Rappi median ETA is {eta_summary.loc[eta_summary['platform'] == 'rappi', 'median_eta_minutes'].iloc[0]} min versus {eta_summary.loc[eta_summary['platform'] == 'uber_eats', 'median_eta_minutes'].iloc[0]} min for Uber Eats, so Rappi is taking more than twice as long on the median delivery promise.",
    "When users see a delivery promise that is materially slower than the leading alternative, many of them will switch platforms before checkout. That hurts conversion even when Rappi is price-competitive on the basket itself.",
    "Improve dispatch and order allocation logic in the worst-performing zones first, rebalance rider supply against demand peaks, and review store-to-courier assignment rules so the ETA promise converges toward Uber Eats instead of staying 2x slower.",
)




,platform,eta_snapshots,avg_eta_minutes,median_eta_minutes,platform_label
0,didi_food,0,NaN,NaN,DiDi Food
1,rappi,41,35.39,34.0,Rappi
2,uber_eats,38,13.87,12.5,Uber Eats


,zone_name,rappi_eta,uber_eta,eta_gap_rappi_vs_uber
0,Lomas de Chapultepec,43.5,10.0,33.5
1,Polanco,43.0,14.5,28.5
2,San Rafael,36.0,10.0,26.0
3,Mixcoac,34.0,10.0,24.0
4,Santa Fe,34.0,10.0,24.0
5,Azcapotzalco,37.0,13.5,23.5
6,Condesa,36.0,13.0,23.0
7,Iztapalapa Centro,34.0,11.5,22.5
8,Tacubaya,34.0,11.5,22.5
9,Doctores,34.0,13.0,21.0


## 7. Promotional Strategy

The warehouse captures promo intensity directly through `discount_count`. Promo *type* is inferred here from keywords visible in matched tracked products, so it should be read as a directional pattern, not as a full promotion catalog.


In [7]:
promo_intensity = q("""
select platform,
       round(avg(discount_count), 2) as avg_discount_count,
       round(median(discount_count), 2) as median_discount_count,
       round(avg(case when discount_count = 0 then 1 else 0 end) * 100, 1) as zero_discount_snapshot_pct
from snapshots
group by 1
order by 1
""")
promo_intensity["platform_label"] = promo_intensity["platform"].map(pretty_platform)
show_frame(promo_intensity)

promo_types = q(r"""
with matched as (
    select platform,
           lower(coalesce(product_name, '') || ' ' || coalesce(description, '')) as text
    from product_matches
), exploded as (
    select platform, 'bundle_combo' as promo_type from matched where regexp_matches(text, 'combo')
    union all
    select platform, 'freebie' as promo_type from matched where regexp_matches(text, 'gratis')
    union all
    select platform, 'gift_or_promotional_item' as promo_type from matched where regexp_matches(text, 'promocional|vaso')
    union all
    select platform, 'multipack' as promo_type from matched where regexp_matches(text, '2x1|3x2')
    union all
    select platform, 'percent_discount' as promo_type from matched where regexp_matches(text, '[0-9]+\\s*%')
)
select platform, promo_type, count(*) as matched_rows
from exploded
group by 1, 2
order by 1, 3 desc, 2
""")

promo_types["platform_label"] = promo_types["platform"].map(pretty_platform)
promo_types["share_pct"] = (
    promo_types["matched_rows"]
    / promo_types.groupby("platform")["matched_rows"].transform("sum")
    * 100
).round(1)
show_frame(promo_types)

promo_fig = px.bar(
    promo_types,
    x="platform_label",
    y="share_pct",
    color="promo_type",
    barmode="stack",
    labels={"platform_label": "Platform", "share_pct": "Share of inferred promo-type signals (%)"},
)
promo_fig.update_layout(title="Inferred Promotion Mix from Matched Product Text", height=420)
show_figure(promo_fig)

rappi_discounts = promo_intensity[promo_intensity["platform"] == "rappi"]["avg_discount_count"].iloc[0]
uber_discounts = promo_intensity[promo_intensity["platform"] == "uber_eats"]["avg_discount_count"].iloc[0]
didi_discounts = promo_intensity[promo_intensity["platform"] == "didi_food"]["avg_discount_count"].iloc[0]

promo_type_explanations = [
    "bundle_combo: packaged deal that bundles several items into one offer.",
    "freebie: a free extra item attached to the purchase, such as papotas or cajeta gratis.",
    "gift_or_promotional_item: a promotional add-on or gift, such as a vaso promocional.",
    "percent_discount: matched text that explicitly signals a percentage-off deal.",
    "multipack: multi-buy logic such as 2x1 or 3x2.",
]
show_note("Promo type quick guide", promo_type_explanations)

show_insight(
    "Promotional strategy insight",
    f"Visible promo intensity on Rappi is only {rappi_discounts} average discount signals per snapshot, versus {uber_discounts} on Uber Eats and {didi_discounts} on DiDi Food. Competitors are putting more promotional messages in front of the user, especially through bundle combos and freebies.",
    "A weaker promo layer reduces perceived value even when the base price is competitive. Users compare not only price, but also how much extra value each platform appears to be giving them.",
    "Increase promo depth on the most visible mechanics first: bundle combos, freebie-style adds, and zone-priority campaigns where Rappi also has ETA or availability disadvantages. The goal is not just more promos, but more visible promos on the baskets users actually compare.",
)


,platform,avg_discount_count,median_discount_count,zero_discount_snapshot_pct,platform_label
0,didi_food,3.50,3.5,50.0,DiDi Food
1,rappi,1.82,2.0,9.1,Rappi
2,uber_eats,3.73,4.0,37.5,Uber Eats


,platform,promo_type,matched_rows,platform_label,share_pct
0,didi_food,bundle_combo,132,DiDi Food,50.0
1,didi_food,freebie,88,DiDi Food,33.3
2,didi_food,gift_or_promotional_item,44,DiDi Food,16.7
3,rappi,bundle_combo,84,Rappi,62.7
4,rappi,gift_or_promotional_item,33,Rappi,24.6
5,rappi,freebie,17,Rappi,12.7
6,uber_eats,bundle_combo,118,Uber Eats,66.3
7,uber_eats,freebie,60,Uber Eats,33.7


## 8. Comparison Across Platforms

This section compares cross-platform availability. Higher availability is better because it means more stores are open and the platform is operationally more reliable for the user.


In [8]:
availability_summary = q("""
select platform,
       count(*) as snapshots,
       sum(case when is_open then 1 else 0 end) as open_snapshots,
       round(avg(case when is_open then 1 else 0 end) * 100, 1) as open_rate_pct
from snapshots
group by 1
order by 1
""")

availability_summary["platform_label"] = availability_summary["platform"].map(pretty_platform)
show_frame(availability_summary)

availability_fig = px.bar(
    availability_summary,
    x="platform_label",
    y="open_rate_pct",
    color="platform_label",
    labels={"platform_label": "Platform", "open_rate_pct": "Availability / open rate (%)"},
)
availability_fig.update_layout(title="Availability Comparison Across Platforms", height=360, showlegend=False)
show_figure(availability_fig)

lowest_availability = availability_summary.sort_values("open_rate_pct", ascending=True).iloc[0]
highest_availability = availability_summary.sort_values("open_rate_pct", ascending=False).iloc[0]


availability_cases = q("""
with rappi_missing as (
    select zone_name, restaurant_brand, error
    from snapshots
    where platform = 'rappi'
      and coalesce(is_open, false) = false
)
select r.zone_name,
       r.restaurant_brand,
       max(case when s.platform = 'uber_eats' then coalesce(s.is_open, false)::int end) as uber_open,
       max(case when s.platform = 'didi_food' then coalesce(s.is_open, false)::int end) as didi_open,
       max(r.error) as error
from rappi_missing r
left join snapshots s using (zone_name, restaurant_brand)
group by 1, 2
order by 1, 2
""")
show_frame(availability_cases)

missing_coverage_labels = [
    f"{row.zone_name} - {row.restaurant_brand}"
    for row in availability_cases.itertuples()
]

show_insight(
    "Availability insight",
    f"Rappi has 3 no-coverage cases in the current sample: {join_values(missing_coverage_labels)}.",
    "These no-coverage cases mean Rappi is losing competitiveness exactly where the user expects the same restaurant to be orderable on competing platforms. In those zones, the order demand can migrate directly to Uber Eats or DiDi Food.",
    "Treat these cases as urgent coverage-recovery work: verify store onboarding and geocoverage for Milpa Alta and Tláhuac, audit how location resolution is being matched to store coverage, and prioritize commercial or operational fixes so those restaurant-zone combinations stop leaking demand to competitors.",
    extra=["Peer check: Uber Eats and DiDi Food both show those same restaurant-zone pairs as open, so this is a Rappi-specific coverage gap rather than a market-wide closure."],
)


,platform,snapshots,open_snapshots,open_rate_pct,platform_label
0,didi_food,44,44.0,100.0,DiDi Food
1,rappi,44,41.0,93.2,Rappi
2,uber_eats,40,40.0,100.0,Uber Eats


,zone_name,restaurant_brand,uber_open,didi_open,error
0,Milpa Alta,Domino's Pizza,1,1,RappiClientError: No coverage for Domino's Pizza at Milpa Alta
1,Milpa Alta,McDonald's,1,1,RappiClientError: No coverage for McDonald's at Milpa Alta
2,Tláhuac,McDonald's,1,1,RappiClientError: No coverage for McDonald's at Tláhuac
